# Global Evaluator Audit

This notebook is the single global inspection notebook for CALE evaluator results.

本 notebook 用来把所有结果放在同一个全局检测框架里，而不是只看 strong evaluator。

It explicitly separates three layers that should not be mixed:

1. **Target model**: the model that generated the candidate response.
2. **Evaluator backend**: the judge implementation/model used to score the response.
3. **Evaluator variant**: the evaluation protocol or CALE variant.

中文速读：`model_name` 是被评价的回答模型，不是 evaluator；`evaluator_backend_*` 是谁在评价；`variant` 是用什么评价结构。

## 0. Concept Dictionary

| Layer | Meaning | Source in this notebook | Examples |
|---|---|---|---|
| `target_model` | response-generating model being evaluated | copied from CSV `model_name` | Qwen2.5-1.5B, Llama-3.2-1B |
| `evaluator_backend` | judge backend/model that performs scoring | injected from `RUNS` registry | rule-based heuristic, Qwen2.5-7B HF, DeepSeek V4-Pro |
| `evaluator_variant` | scoring protocol / evaluation design | copied from CSV `variant` | direct_llm_judge, generic_cale, attack_aware_cale, full_attack_aware_cale |

Do not describe `full_attack_aware_cale` as an evaluator backend. It is a variant/protocol.

不要把 `full_attack_aware_cale` 叫成 evaluator backend；它是 evaluator variant。

In [ ]:
from pathlib import Path
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analyze_behavior_matrix import numeric_behavior_columns, run_pca
from visualize_behavior_matrix import (
    CORE_BEHAVIOR_COLUMNS,
    PROXY_COLUMNS,
    existing_numeric_columns,
    save_heatmap,
    save_missingness_plot,
    summarize_by,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

OUTDIR = Path('figures/global_evaluator_audit')
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TARGET_SPLITS = {'target_qwen_only', 'target_llama_only'}
CALE_VARIANTS = {'generic_cale', 'attack_aware_cale', 'full_attack_aware_cale'}
FULL_CALE_VARIANT = 'full_attack_aware_cale'
DIRECT_VARIANT = 'direct_llm_judge'

RUNS = [
    {
        'run_id': 'heuristic_full',
        'evaluator_backend_family': 'heuristic',
        'evaluator_backend_model': 'heuristic/default',
        'evaluator_backend_label': 'Rule-based heuristic judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json'),
        'expected_rows': 239976,
        'expected_variants': ['baseline_binary', 'baseline_likert', 'direct_trustllm_heuristic', 'generic_cale', 'attack_aware_cale', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only', 'target_llama_only'],
        'subset_policy': 'full_neutral_fever_qwen_and_llama_targets',
        'notes': 'Main full-sample rule-based result; primary source for PCA/CFA.',
    },
    {
        'run_id': 'qwen25_7b_limit1000',
        'evaluator_backend_family': 'hf_local',
        'evaluator_backend_model': 'Qwen/Qwen2.5-7B-Instruct',
        'evaluator_backend_label': 'Qwen2.5-7B local HF judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_from_fixed_response_file; likely qwen_target_only unless a later balanced run is used',
        'notes': 'Strong local evaluator subset; use for backend robustness, not full target dominance.',
    },
    {
        'run_id': 'deepseek_v4_pro_limit1000',
        'evaluator_backend_family': 'api',
        'evaluator_backend_model': 'deepseek-v4-pro',
        'evaluator_backend_label': 'DeepSeek V4-Pro API judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_from_fixed_response_file; currently verified as qwen_target_only',
        'notes': 'Strong API evaluator subset; use for backend robustness, not full target dominance.',
    },
]

METADATA_REQUIRED = ['id', 'model_name', 'variant', 'reference_label', 'dataset']
OUTCOME_COLUMNS = ['final_score', 'quality_label', 'uncertainty']
BEHAVIOR_COLUMNS = CORE_BEHAVIOR_COLUMNS + PROXY_COLUMNS

run_registry = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in run.items()} for run in RUNS])
run_registry.to_csv(OUTDIR / 'run_registry.csv', index=False)
run_registry

## 1. Helper Functions

These helpers enforce the layer separation. In particular, `target_model` is copied from `model_name`, while evaluator backend columns are injected from the run registry.

这些 helper 的作用是强制分清三层：target model 来自 CSV，evaluator backend 来自 notebook registry。

In [ ]:
def target_split_name(model_name):
    value = str(model_name).lower()
    if 'qwen' in value:
        return 'target_qwen_only'
    if 'llama' in value:
        return 'target_llama_only'
    return 'target_other'


def target_coverage_label(splits):
    present = set(splits)
    if EXPECTED_TARGET_SPLITS <= present:
        return 'pooled_qwen_llama'
    if present == {'target_qwen_only'}:
        return 'qwen_target_only'
    if present == {'target_llama_only'}:
        return 'llama_target_only'
    if not present:
        return 'missing_file_or_no_target_model'
    return 'partial_or_unknown'


def existing_behavior_columns(df):
    return existing_numeric_columns(df, CORE_BEHAVIOR_COLUMNS) + existing_numeric_columns(df, PROXY_COLUMNS)


def load_run(run):
    path = run['behavior_matrix_path']
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df = df.copy()
    df['target_model'] = df['model_name'].astype(str) if 'model_name' in df.columns else 'unknown_target_model'
    df['target_split'] = df['target_model'].map(target_split_name)
    df['evaluator_variant'] = df['variant'].astype(str) if 'variant' in df.columns else 'unknown_variant'
    for key in ['run_id', 'evaluator_backend_family', 'evaluator_backend_model', 'evaluator_backend_label', 'subset_policy']:
        df[key] = run[key]
    return df


def score_column(df):
    if 'final_score' in df.columns:
        return 'final_score'
    if 'score' in df.columns:
        return 'score'
    return None


def save_bar(table, path, title, ylabel='value', rotation=25):
    if table.empty:
        return
    fig, ax = plt.subplots(figsize=(max(7, 0.65 * len(table.index)), 4.8))
    table.plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=rotation)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=240)
    plt.show()


def top_loading_records(loadings, run_id, backend_label, target_split='pooled_all_available_targets', n=8):
    rows = []
    for component in loadings.columns:
        ordered = loadings[component].sort_values(key=lambda s: s.abs(), ascending=False)
        for rank, (variable, loading) in enumerate(ordered.head(n).items(), start=1):
            rows.append({
                'run_id': run_id,
                'evaluator_backend_label': backend_label,
                'target_split': target_split,
                'component': component,
                'rank': rank,
                'variable': variable,
                'loading': loading,
                'absolute_loading': abs(loading),
            })
    return rows


def run_pca_summary(df, run_id, backend_label, target_split, n_components=4, max_missing_share=0.5):
    columns = numeric_behavior_columns(df, include_final_score=False)
    selected = df[columns].apply(pd.to_numeric, errors='coerce') if columns else pd.DataFrame()
    loadings, variance, scores = run_pca(selected, n_components=n_components, max_missing_share=max_missing_share)
    return {
        'run_id': run_id,
        'evaluator_backend_label': backend_label,
        'target_split': target_split,
        'rows': len(df),
        'columns_used': ', '.join(loadings.index.tolist()),
        'pc1_variance': variance.loc[variance['component'].eq('PC1'), 'explained_variance_ratio'].iloc[0] if 'PC1' in set(variance['component']) else np.nan,
        'pc1_pc4_cumulative_variance': variance['explained_variance_ratio'].sum(),
        'pc1_top_variables': ', '.join(loadings['PC1'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC1' in loadings else '',
        'pc2_top_variables': ', '.join(loadings['PC2'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC2' in loadings else '',
    }, loadings, variance, scores


def paired_direct_vs_cale(df):
    score_col = score_column(df)
    if score_col is None:
        return pd.DataFrame()
    needed = [DIRECT_VARIANT, FULL_CALE_VARIANT]
    subset = df[df['evaluator_variant'].isin(needed)].copy()
    if subset.empty:
        return pd.DataFrame()
    key_cols = ['run_id', 'evaluator_backend_label', 'target_model', 'id']
    if not all(col in subset.columns for col in key_cols):
        return pd.DataFrame()
    wide = subset.pivot_table(index=key_cols, columns='evaluator_variant', values=score_col, aggfunc='mean')
    if not all(col in wide.columns for col in needed):
        return pd.DataFrame()
    wide = wide.dropna(subset=needed).reset_index()
    wide['full_minus_direct_score'] = wide[FULL_CALE_VARIANT] - wide[DIRECT_VARIANT]
    return wide


## 2. File, Provenance, And Coverage Audit

This is the first table to inspect on the server. It tells us which files exist, whether the row counts match, which target models are covered, and whether target splits are missing.

服务器上先看这一张表：它回答文件是否存在、行数是否对、覆盖了哪些 target models、有没有缺 target split。

In [ ]:
matrices = {}
audit_rows = []

for run in RUNS:
    df = load_run(run)
    if df is not None:
        matrices[run['run_id']] = df
        target_models = sorted(df['target_model'].dropna().astype(str).unique())
        target_splits = sorted(df['target_split'].dropna().astype(str).unique())
        variants = sorted(df['evaluator_variant'].dropna().astype(str).unique())
        missing_splits = sorted(set(run['expected_target_splits']) - set(target_splits))
        missing_variants = sorted(set(run['expected_variants']) - set(variants))
    else:
        target_models, target_splits, variants, missing_splits, missing_variants = [], [], [], list(run['expected_target_splits']), list(run['expected_variants'])

    audit_rows.append({
        'run_id': run['run_id'],
        'evaluator_backend_label': run['evaluator_backend_label'],
        'evaluator_backend_family': run['evaluator_backend_family'],
        'evaluator_backend_model': run['evaluator_backend_model'],
        'path': str(run['behavior_matrix_path']),
        'exists': run['behavior_matrix_path'].exists(),
        'rows': len(df) if df is not None else 0,
        'expected_rows': run['expected_rows'],
        'row_count_ok': (len(df) == run['expected_rows']) if df is not None else False,
        'evaluator_variants': ', '.join(variants),
        'missing_expected_variants': ', '.join(missing_variants),
        'target_models': ', '.join(target_models),
        'target_splits_present': ', '.join(target_splits),
        'target_splits_missing': ', '.join(missing_splits),
        'target_coverage': target_coverage_label(target_splits),
        'coverage_warning': bool(missing_splits),
        'subset_policy': run['subset_policy'],
        'notes': run['notes'],
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTDIR / 'global_file_audit.csv', index=False)
audit

## 3. Layer Dictionary And Validation Dashboard

These checks are designed to catch naming mistakes such as treating `full_attack_aware_cale` as a backend or treating `Qwen/Qwen2.5-1.5B-Instruct` as a judge.

这些 check 用来防止最容易犯的命名错误：把 full CALE 当 backend，或者把 target Qwen 当 evaluator。

In [ ]:
layer_dictionary = pd.DataFrame([
    {
        'layer': 'target_model',
        'meaning': 'model that generated the candidate response',
        'source_column': 'model_name copied to target_model',
        'examples': 'Qwen/Qwen2.5-1.5B-Instruct; meta-llama/Llama-3.2-1B-Instruct',
        'do_not_mix_with': 'evaluator_backend_model',
    },
    {
        'layer': 'evaluator_backend',
        'meaning': 'judge backend or judge model used to score the response',
        'source_column': 'injected from RUNS registry',
        'examples': 'heuristic/default; Qwen/Qwen2.5-7B-Instruct; deepseek-v4-pro',
        'do_not_mix_with': 'target_model or evaluator_variant',
    },
    {
        'layer': 'evaluator_variant',
        'meaning': 'evaluation protocol or scoring structure',
        'source_column': 'variant copied to evaluator_variant',
        'examples': 'direct_llm_judge; generic_cale; attack_aware_cale; full_attack_aware_cale',
        'do_not_mix_with': 'evaluator_backend_model',
    },
])
layer_dictionary.to_csv(OUTDIR / 'layer_dictionary.csv', index=False)

validation_rows = []
for run_id, df in matrices.items():
    required_cols = METADATA_REQUIRED + ['evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant', 'target_model', 'target_split']
    missing_required = [col for col in required_cols if col not in df.columns]
    key_cols = ['dataset', 'id', 'target_model', 'evaluator_backend_model', 'evaluator_variant']
    if all(col in df.columns for col in key_cols):
        duplicate_count = int(df.duplicated(subset=key_cols).sum())
    else:
        duplicate_count = np.nan
    variant_values = set(df['evaluator_variant'].dropna().astype(str)) if 'evaluator_variant' in df.columns else set()
    backend_like_variants = sorted([v for v in variant_values if any(token in v.lower() for token in ['deepseek', 'qwen2.5-7b', 'heuristic full evaluator'])])
    target_models = set(df['target_model'].dropna().astype(str)) if 'target_model' in df.columns else set()
    backend_model = str(df['evaluator_backend_model'].iloc[0]) if 'evaluator_backend_model' in df.columns and len(df) else ''
    backend_in_target = backend_model in target_models

    validation_rows.append({
        'run_id': run_id,
        'status': 'pass' if not missing_required and duplicate_count == 0 and not backend_like_variants and not backend_in_target else 'review',
        'missing_required_columns': ', '.join(missing_required),
        'duplicate_row_key_count': duplicate_count,
        'backend_like_variant_values': ', '.join(backend_like_variants),
        'evaluator_backend_model_appears_as_target_model': backend_in_target,
        'message': 'Review any non-pass row before interpreting global comparisons.',
    })

validation_dashboard = pd.DataFrame(validation_rows)
validation_dashboard.to_csv(OUTDIR / 'global_validation_dashboard.csv', index=False)
display(layer_dictionary)
validation_dashboard

## 4. Schema And Missingness Audit

This checks which behavior variables are available for each run. Direct judge rows usually lack CALE construct subscores, so construct-level analyses must filter to CALE variants.

这里检查每个 run 有哪些 behavior variables。direct judge 通常没有 CALE construct subscores，所以 construct-level 分析必须过滤 CALE variants。

In [ ]:
missingness_rows = []
for run_id, df in matrices.items():
    for col in METADATA_REQUIRED + OUTCOME_COLUMNS + BEHAVIOR_COLUMNS:
        missingness_rows.append({
            'run_id': run_id,
            'evaluator_backend_label': df['evaluator_backend_label'].iloc[0],
            'column': col,
            'exists': col in df.columns,
            'non_null_share': pd.to_numeric(df[col], errors='coerce').notna().mean() if col in df.columns and col in BEHAVIOR_COLUMNS + ['final_score', 'uncertainty'] else (df[col].notna().mean() if col in df.columns else 0.0),
        })

schema_missingness = pd.DataFrame(missingness_rows)
schema_missingness.to_csv(OUTDIR / 'schema_missingness_audit.csv', index=False)

if not schema_missingness.empty:
    availability = schema_missingness.pivot(index='evaluator_backend_label', columns='column', values='non_null_share').fillna(0)
    availability.to_csv(OUTDIR / 'schema_availability_pivot.csv')
    behavior_availability = availability[[c for c in BEHAVIOR_COLUMNS if c in availability.columns]]
    if not behavior_availability.empty:
        save_heatmap(behavior_availability, OUTDIR / 'behavior_variable_availability_by_backend.png', 'Behavior Variable Availability by Evaluator Backend', vmin=0, vmax=1)
    display(availability.round(2))
else:
    print('No matrices loaded yet.')

## 5. Cross-Product Coverage

This table tells us the actual experimental coverage across target model, evaluator backend, and evaluator variant.

这张表只看设计覆盖，不解释效果：target model × evaluator backend × evaluator variant 是否齐。

In [ ]:
combined = pd.concat(matrices.values(), ignore_index=True, sort=False) if matrices else pd.DataFrame()
if not combined.empty:
    combined.to_csv(OUTDIR / 'combined_available_behavior_matrix.csv', index=False)
    coverage = combined.groupby(['evaluator_backend_label', 'target_model', 'evaluator_variant'], dropna=False).size().rename('rows').reset_index()
    coverage.to_csv(OUTDIR / 'cross_product_coverage_long.csv', index=False)
    coverage_pivot = coverage.pivot_table(index='evaluator_backend_label', columns=['target_model', 'evaluator_variant'], values='rows', fill_value=0, aggfunc='sum')
    coverage_pivot.to_csv(OUTDIR / 'cross_product_coverage.csv')
    display(coverage_pivot)
else:
    coverage = pd.DataFrame()
    coverage_pivot = pd.DataFrame()
    print('No matrices loaded yet. Run this notebook on the server after outputs are available.')

## 6. Outcome Summary By Three Layers

Outcome-level summaries can include direct judge and CALE variants. Backend-to-backend absolute scores are descriptive because scoring scales may differ.

Outcome 层可以比较 direct judge 和 CALE variants；但不同 backend 的绝对分数不是 calibrated leaderboard。

In [ ]:
outcome_rows = []
if not combined.empty:
    sc = score_column(combined)
    group_cols = ['evaluator_backend_label', 'target_model', 'evaluator_variant']
    for keys, group in combined.groupby(group_cols, dropna=False):
        backend_label, target_model, variant = keys
        row = {
            'evaluator_backend_label': backend_label,
            'target_model': target_model,
            'evaluator_variant': variant,
            'rows': len(group),
        }
        if sc:
            scores = pd.to_numeric(group[sc], errors='coerce')
            row['mean_final_score'] = scores.mean()
            row['std_final_score'] = scores.std()
        if 'uncertainty' in group.columns:
            row['mean_uncertainty'] = pd.to_numeric(group['uncertainty'], errors='coerce').mean()
        if 'quality_label' in group.columns:
            row['quality_label_distribution'] = group['quality_label'].astype(str).value_counts(normalize=True).round(3).to_dict()
        outcome_rows.append(row)

outcome_summary = pd.DataFrame(outcome_rows)
outcome_summary.to_csv(OUTDIR / 'outcome_summary_three_layers.csv', index=False)
outcome_summary

In [ ]:
if not outcome_summary.empty and 'mean_final_score' in outcome_summary.columns:
    score_pivot = outcome_summary.pivot_table(index='evaluator_backend_label', columns='evaluator_variant', values='mean_final_score', aggfunc='mean')
    score_pivot.to_csv(OUTDIR / 'mean_score_by_backend_variant.csv')
    save_bar(score_pivot, OUTDIR / 'mean_score_by_backend_variant.png', 'Mean Score by Evaluator Backend and Variant', ylabel='mean final score')
    display(score_pivot.round(3))

## 7. Direct Judge Versus Full CALE Delta

This compares `direct_llm_judge` and `full_attack_aware_cale` within the same evaluator backend and target model, preferably on shared ids.

这里比较的是同一个 backend、同一个 target model 内，direct judge 和 full CALE 的差异。

In [ ]:
paired_tables = []
for run_id, df in matrices.items():
    paired = paired_direct_vs_cale(df)
    if not paired.empty:
        paired_tables.append(paired)

paired_delta_rows = []
if paired_tables:
    paired_all = pd.concat(paired_tables, ignore_index=True)
    paired_all.to_csv(OUTDIR / 'paired_direct_vs_full_cale_rows.csv', index=False)
    for keys, group in paired_all.groupby(['run_id', 'evaluator_backend_label', 'target_model'], dropna=False):
        run_id, backend_label, target_model = keys
        paired_delta_rows.append({
            'run_id': run_id,
            'evaluator_backend_label': backend_label,
            'target_model': target_model,
            'n_shared_ids': len(group),
            'direct_mean_score': group[DIRECT_VARIANT].mean(),
            'full_cale_mean_score': group[FULL_CALE_VARIANT].mean(),
            'full_minus_direct_mean_score': group['full_minus_direct_score'].mean(),
        })

direct_vs_full_delta = pd.DataFrame(paired_delta_rows)
direct_vs_full_delta.to_csv(OUTDIR / 'direct_vs_full_cale_delta.csv', index=False)
direct_vs_full_delta

## 8. Construct Profile By Backend, Target, And Variant

Construct profiles use CALE behavior columns. Direct judge rows are excluded when they lack construct variables.

Construct profile 使用 CALE behavior columns；没有 construct variables 的 direct judge rows 不进入这一步。

In [ ]:
profile_rows = []
if not combined.empty:
    cols = existing_behavior_columns(combined)
    cale_like = combined[combined['evaluator_variant'].isin(CALE_VARIANTS)].copy()
    for keys, group in cale_like.groupby(['evaluator_backend_label', 'target_model', 'evaluator_variant'], dropna=False):
        backend_label, target_model, variant = keys
        row = {
            'evaluator_backend_label': backend_label,
            'target_model': target_model,
            'evaluator_variant': variant,
            'rows': len(group),
        }
        for col in cols:
            row[col] = pd.to_numeric(group[col], errors='coerce').mean()
        profile_rows.append(row)

construct_profile = pd.DataFrame(profile_rows)
construct_profile.to_csv(OUTDIR / 'construct_profile_backend_target_variant.csv', index=False)
construct_profile.head(20)

In [ ]:
if not combined.empty:
    full_combined = combined[combined['evaluator_variant'].eq(FULL_CALE_VARIANT)].copy()
    full_cols = existing_behavior_columns(full_combined)
    if not full_combined.empty and full_cols:
        full_backend_profile = summarize_by(full_combined, 'evaluator_backend_label', full_cols)
        full_backend_profile.to_csv(OUTDIR / 'full_cale_construct_profile_by_backend.csv')
        save_heatmap(full_backend_profile, OUTDIR / 'full_cale_construct_profile_by_backend.png', 'Full CALE Construct Profile by Evaluator Backend')
        display(full_backend_profile.round(3))

## 9. Backend Alignment On Shared Rows

This checks whether different evaluator backends agree on the same target response and evaluator variant. It is backend robustness, not target-model performance.

这里看不同 evaluator backend 对同一条 target response、同一种 evaluator variant 是否一致。

In [ ]:
alignment_rows = []
if not combined.empty:
    sc = score_column(combined)
    if sc and 'id' in combined.columns:
        align = combined[['run_id', 'evaluator_backend_label', 'target_model', 'id', 'evaluator_variant', sc]].copy()
        align[sc] = pd.to_numeric(align[sc], errors='coerce')
        for variant, variant_df in align.groupby('evaluator_variant', dropna=False):
            wide = variant_df.pivot_table(index=['target_model', 'id', 'evaluator_variant'], columns='evaluator_backend_label', values=sc, aggfunc='mean')
            for left, right in itertools.combinations(wide.columns, 2):
                pair = wide[[left, right]].dropna()
                if len(pair) < 3:
                    continue
                alignment_rows.append({
                    'evaluator_variant': variant,
                    'backend_left': left,
                    'backend_right': right,
                    'n_shared_rows': len(pair),
                    'pearson': pair[left].corr(pair[right], method='pearson'),
                    'spearman': pair[left].corr(pair[right], method='spearman'),
                    'mae': (pair[left] - pair[right]).abs().mean(),
                    'mean_signed_difference_left_minus_right': (pair[left] - pair[right]).mean(),
                })

backend_alignment = pd.DataFrame(alignment_rows)
backend_alignment.to_csv(OUTDIR / 'backend_pair_score_alignment.csv', index=False)
backend_alignment

## 10. Target-Model Dominance Check

For each evaluator backend, rerun PCA on `full_attack_aware_cale` rows for all available targets and for each target split. If a backend only has Qwen target rows, the notebook labels that as Qwen-only instead of pretending it is pooled evidence.

每个 backend 内只用 `full_attack_aware_cale` 重新跑 PCA。如果某个 backend 只有 Qwen target，这里会标成 Qwen-only，不会假装它覆盖两个 target。

In [ ]:
pca_rows = []
pca_loading_rows = []

for run_id, df in matrices.items():
    if 'evaluator_variant' not in df.columns:
        continue
    full = df[df['evaluator_variant'].eq(FULL_CALE_VARIANT)].copy()
    if full.empty:
        continue
    backend_label = full['evaluator_backend_label'].iloc[0]
    split_frames = [('pooled_all_available_targets', full)]
    for split_name, split_df in full.groupby('target_split', dropna=False):
        split_frames.append((split_name, split_df.copy()))
    for split_name, split_df in split_frames:
        if len(split_df) < 5:
            pca_rows.append({
                'run_id': run_id,
                'evaluator_backend_label': backend_label,
                'target_split': split_name,
                'rows': len(split_df),
                'error': 'too few rows for PCA',
            })
            continue
        try:
            row, loadings, variance, scores = run_pca_summary(split_df, run_id, backend_label, split_name)
        except ValueError as exc:
            pca_rows.append({
                'run_id': run_id,
                'evaluator_backend_label': backend_label,
                'target_split': split_name,
                'rows': len(split_df),
                'error': str(exc),
            })
            continue
        pca_rows.append(row)
        loadings.to_csv(OUTDIR / f'{run_id}_{split_name}_full_cale_pca_loadings.csv')
        variance.to_csv(OUTDIR / f'{run_id}_{split_name}_full_cale_pca_explained_variance.csv', index=False)
        pca_loading_rows.extend(top_loading_records(loadings, run_id, backend_label, split_name))

target_model_dominance_pca = pd.DataFrame(pca_rows)
target_model_dominance_loadings = pd.DataFrame(pca_loading_rows)
target_model_dominance_pca.to_csv(OUTDIR / 'target_model_dominance_pca_summary.csv', index=False)
target_model_dominance_loadings.to_csv(OUTDIR / 'target_model_dominance_pca_top_loadings.csv', index=False)
target_model_dominance_pca

In [ ]:
if not target_model_dominance_pca.empty and 'pc1_pc4_cumulative_variance' in target_model_dominance_pca.columns:
    dominance_var = target_model_dominance_pca.pivot_table(
        index='evaluator_backend_label',
        columns='target_split',
        values='pc1_pc4_cumulative_variance',
        aggfunc='first',
    )
    dominance_var.to_csv(OUTDIR / 'target_model_dominance_variance_pivot.csv')
    save_bar(dominance_var, OUTDIR / 'target_model_dominance_variance.png', 'PC1-PC4 Variance by Backend and Target-Model Split', ylabel='PC1-PC4 cumulative variance')
    display(dominance_var.round(3))

## 11. CALE Variant Ladder Within The Heuristic Full Run

The heuristic full run is the only current result that contains the full CALE ladder (`generic_cale`, `attack_aware_cale`, `full_attack_aware_cale`) over both target models.

目前只有 heuristic full run 覆盖完整 CALE ladder，所以 variant ladder 的主分析应该放在这里。

In [ ]:
ladder_rows = []
heuristic = matrices.get('heuristic_full')
if heuristic is not None:
    cols = existing_behavior_columns(heuristic)
    ladder = heuristic[heuristic['evaluator_variant'].isin(CALE_VARIANTS)].copy()
    for keys, group in ladder.groupby(['target_split', 'evaluator_variant'], dropna=False):
        target_split, variant = keys
        row = {'target_split': target_split, 'evaluator_variant': variant, 'rows': len(group)}
        for col in cols:
            row[col] = pd.to_numeric(group[col], errors='coerce').mean()
        ladder_rows.append(row)

variant_ladder = pd.DataFrame(ladder_rows)
variant_ladder.to_csv(OUTDIR / 'heuristic_full_variant_ladder_by_target_split.csv', index=False)
if not variant_ladder.empty:
    display(variant_ladder.round(3))
else:
    print('Heuristic full result not available locally.')

## 12. Proxy-Specific Diagnostics

Proxy variables are reference-label-specific. They should be interpreted only where the corresponding FEVER label applies, not as general scores across all rows.

Proxy 变量是 reference-label-specific，不能把非对应 label 的缺失当成 0。

In [ ]:
proxy_specs = {
    'NOT ENOUGH INFO': 'nei_uncertainty_failure_proxy',
    'REFUTES': 'refutes_correction_credit_proxy',
    'SUPPORTS': 'supports_status_failure_proxy',
}
proxy_rows = []
if not combined.empty and 'reference_label' in combined.columns:
    for label, proxy in proxy_specs.items():
        if proxy not in combined.columns:
            continue
        subset = combined[combined['reference_label'].eq(label)].copy()
        for keys, group in subset.groupby(['evaluator_backend_label', 'target_model', 'evaluator_variant'], dropna=False):
            backend_label, target_model, variant = keys
            proxy_rows.append({
                'reference_label': label,
                'proxy': proxy,
                'evaluator_backend_label': backend_label,
                'target_model': target_model,
                'evaluator_variant': variant,
                'rows': len(group),
                'mean_proxy': pd.to_numeric(group[proxy], errors='coerce').mean(),
                'non_null_share': pd.to_numeric(group[proxy], errors='coerce').notna().mean(),
            })

proxy_diagnostics = pd.DataFrame(proxy_rows)
proxy_diagnostics.to_csv(OUTDIR / 'proxy_specific_diagnostics.csv', index=False)
proxy_diagnostics.head(30)

## 13. Paper-Facing Global Summary

This final cell writes a compact markdown summary. Treat it as a draft note, not as final paper prose.

最后这一步生成一个简短 markdown summary，作为写作草稿，不是最终论文语言。

In [ ]:
summary_lines = []
summary_lines.append('# Global Evaluator Audit Summary')
summary_lines.append('')
summary_lines.append('Layer reminder: `target_model` is the response-generating model; `evaluator_backend_label` is the judge backend; `evaluator_variant` is the evaluation protocol.')
summary_lines.append('')

if not audit.empty:
    summary_lines.append('## File Audit')
    for _, row in audit.iterrows():
        summary_lines.append(
            f"- {row['evaluator_backend_label']}: exists={row['exists']}, rows={row['rows']}/{row['expected_rows']}, "
            f"coverage={row['target_coverage']}, variants=[{row['evaluator_variants']}]."
        )
    summary_lines.append('')

if 'target_model_dominance_pca' in globals() and not target_model_dominance_pca.empty:
    summary_lines.append('## PCA / Target Dominance')
    for _, row in target_model_dominance_pca.iterrows():
        if pd.notna(row.get('pc1_pc4_cumulative_variance', np.nan)):
            summary_lines.append(
                f"- {row['evaluator_backend_label']} / {row['target_split']}: "
                f"PC1-PC4 variance={row['pc1_pc4_cumulative_variance']:.3f}; "
                f"PC1 top variables={row.get('pc1_top_variables', '')}."
            )
    summary_lines.append('')

summary_lines.append('## Caveats')
summary_lines.append('- Strong evaluator `limit1000` runs may cover only Qwen target responses if they were taken from the beginning of the fixed response JSONL.')
summary_lines.append('- Do not interpret cross-backend absolute scores as a calibrated leaderboard.')
summary_lines.append('- PCA signs are arbitrary; interpret loading magnitudes and variable clusters.')

summary_text = chr(10).join(summary_lines)
(OUTDIR / 'global_interpretation_notes.md').write_text(summary_text, encoding='utf-8')
print(summary_text)
